# NB_patient_360
**Silver → Gold | CareSync Health Network** (per STTM v1.0)

* **Target:** `adb_caresync.gold.patient_360`
* **Grain:** one row per patient
* **Driving table:** `adb_caresync.silver.patients`
* **Joins:** LEFT JOIN `hospitals` (registered_hospital_id), LEFT JOIN `insurance_providers` (insurance_provider_id), LEFT JOIN aggregated `lab_results` (by patient_id)
* **Filters:** none (all patients included)
* **Load:** FULL — CREATE OR REPLACE TABLE ... AS SELECT

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary (STTM v1.0)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                    AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                AS date_of_birth,
  CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                        AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                         AS insurance_provider_id,
  ip.provider_name                                                             AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                     AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                             AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                             AS critical_test_count,
  la.last_test_date
FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h            ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip ON p.insurance_provider_id  = ip.insurance_provider_id
LEFT JOIN lab_agg la                                 ON p.patient_id             = la.patient_id;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM adb_caresync.gold.patient_360)                    AS gold_rows,
  (SELECT COUNT(DISTINCT patient_id) FROM adb_caresync.gold.patient_360)  AS gold_distinct_patients,
  (SELECT COUNT(*) FROM adb_caresync.silver.patients)                     AS silver_patients,
  (SELECT SUM(total_tests) FROM adb_caresync.gold.patient_360)            AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results)                  AS silver_lab_results;

In [0]:
record_count = spark.sql("SELECT COUNT(*) AS record_count FROM adb_caresync.gold.patient_360").first()["record_count"]



dbutils.notebook.exit(record_count)